# Fine-Tune mT5 for Burmese Agricultural Concept NER on Google Colab

This notebook demonstrates how to easily clone the repository, install dependencies, prepare/unzip the dataset, and run the fine-tuning script on a Google Colab GPU instance.

### Models Supported:
- `google/mt5-small` (300M parameters - fast CPU baseline)
- `google/mt5-base` (580M parameters - **Highly Recommended for production and accurate Burmese predictions**)

## Step 1: Check GPU Availability
Make sure you have selected a GPU runtime (**Runtime** > **Change runtime type** > **T4 GPU**).

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"🚀 GPU is available: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("⚠️ No GPU detected. Go to Runtime > Change runtime type and select T4 GPU.")

## Step 2: Clone the Public Repository
We clone the public GitHub repository and change directory into it.

In [ ]:
!git clone https://github.com/linhtutkyawdev/myanBertAgriCNER.git
%cd myanBertAgriCNER

## Step 3: Install Dependencies
Install the packages specified in `pyproject.toml` directly into the system environment.

In [ ]:
!pip install .

## Step 4: Prepare the Dataset
You have two options to set up your processed train/validation/test splits:

### Option A: Unzip a Pre-prepared `processed.zip` (Fastest / Recommended)
If you prepare your dataset splits locally and pushed `processed.zip` to the root of your Git repository, run this block to unzip it directly into the dataset directories.

In [ ]:
# Unzip pre-prepared dataset splits directly into the data directory
!unzip -o processed.zip -d data/

### Option B: Parse & Split Dataset from Raw Chunk Files
If you want to parse and split raw files from scratch on Colab, run this block instead:

In [ ]:
!python scripts/prepare_data.py

## Step 5: Start Fine-Tuning the Model on GPU
Run the training pipeline. The training script will auto-detect the Colab GPU and activate mixed precision (FP16) training with Adafactor optimizer.

You can easily toggle between `google/mt5-small` and `google/mt5-base` by changing `--model_name` below.

In [ ]:
# Train with mt5-base (highly recommended for superior Burmese prediction results)
!python scripts/train_mt5.py --model_name google/mt5-base

# ALTERNATIVE (Uncomment to train with mt5-small instead):
# !python scripts/train_mt5.py --model_name google/mt5-small

## Step 6: Test Model Predictions
Test predictions on any custom Burmese agricultural text.

In [ ]:
!python scripts/predict.py --model mt5 --text "စပါး စိုက်ပျိုး ရာတွင် ဂျစ်ဆန် နှင့် ယူရီးယား ကို ၂ကြိမ် ခွဲ၍သုံးပါ၊၊"

## Step 7: Export Model and Metrics to Google Drive
Mount Google Drive and securely export the fine-tuned model weights, configs, and training/test metrics reports. This step includes automatic directory cleanup to prevent `FileExistsError` and creates a ZIP archive for fast Google Drive file transfers.

In [ ]:
from google.colab import drive
import shutil
import os
import datetime

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define source and destination directories
source_model_dir = "experiments/mt5/best_model"
source_metrics_file = "experiments/mt5/metrics.json"
source_config_file = "experiments/mt5/config.json"

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
drive_target_dir = f"/content/drive/MyDrive/myanbert_cner_mt5_best_model_{timestamp}"
zip_filename = f"/content/drive/MyDrive/myanbert_cner_mt5_best_model_{timestamp}.zip"

# 2. Copy the files to a timestamped folder on Google Drive
if os.path.exists(source_model_dir):
    print(f"Exporting model checkpoints and reports to Google Drive folder: {drive_target_dir}...")
    
    # Copy directory
    shutil.copytree(source_model_dir, os.path.join(drive_target_dir, "best_model"))
    
    # Copy metrics and config files if they exist
    if os.path.exists(source_metrics_file):
        shutil.copy2(source_metrics_file, drive_target_dir)
    if os.path.exists(source_config_file):
        shutil.copy2(source_config_file, drive_target_dir)
        
    print("✅ Folder copy complete!")
    
    # 3. Create a ZIP archive (Google Drive is much faster at transferring single large files than hundreds of small files)
    print(f"Creating a compact ZIP archive at {zip_filename}...")
    shutil.make_archive(
        base_name=drive_target_dir, 
        format='zip', 
        root_dir=source_model_dir
    )
    print("✅ ZIP archive created successfully!")
    
else:
    print(f"❌ Source directory {source_model_dir} not found. Please ensure training finished successfully.")